In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt
import os
import sys
from scipy.io import mmwrite
from scipy.sparse import coo_matrix

In [7]:
all_sample=pd.read_csv('/data/work/file/param_input.csv')
all_sample_dict = dict(zip(all_sample['sample'], all_sample['sample_data']))

In [ ]:
sample_id='sample_tmp'
binsize=100

In [ ]:
positionFile = pd.read_csv(all_sample_dict[sample_id], sep='\t',compression='gzip',comment='#')
positionFile['x'] = (positionFile['x']/binsize).astype(np.uint32)*binsize+int(binsize/2)
positionFile['y'] = (positionFile['y']/binsize).astype(np.uint32)*binsize+int(binsize/2)
positionFile=positionFile.loc[:,['geneID','x','y','MIDCount']]

In [ ]:
result = positionFile.groupby(['geneID', 'x', 'y']).agg({'MIDCount': 'sum'}).reset_index()
result = result.sort_values(by=['x', 'y'], ascending=[True, True])
result[f'bin{binsize}_pos']=result.apply(lambda row: f"bin{binsize}_{row['x']}_{row['y']}", axis=1)

In [22]:
output_dir = f'/data/work/STAGATE/10x/bin100/{sample_id}'
os.makedirs(output_dir, exist_ok=True)

In [10]:
df=result

In [11]:
barcodes = df[f'bin{binsize}_pos'].unique()
with open(os.path.join(output_dir, 'barcodes.tsv'), 'w') as f:
    for barcode in barcodes:
        f.write(f"{barcode}\n")
print('barcodes.tsv has finished') 

barcodes.tsv has finished


In [12]:
genes = df['geneID'].unique()
with open(os.path.join(output_dir, 'genes.tsv'), 'w') as f:
    for gene in genes:
        f.write(f"{gene}\t{gene}\n")
print('genes.tsv has finished')

genes.tsv has finished


In [13]:
gene_dict = {gene: idx for idx, gene in enumerate(genes)}
barcode_dict = {barcode: idx for idx, barcode in enumerate(barcodes)}
rows = df['geneID'].map(gene_dict).values
cols = df[f'bin{binsize}_pos'].map(barcode_dict).values
data = df['MIDCount'].values
matrix = coo_matrix((data, (rows, cols)), shape=(len(genes), len(barcodes)))
with open(os.path.join(output_dir, 'matrix.mtx'), 'w') as f:
    f.write('%%MatrixMarket matrix coordinate integer general\n')
    f.write('%\n')
    f.write(f'{matrix.shape[0]} {matrix.shape[1]} {matrix.nnz}\n')
    for i, j, v in zip(matrix.row, matrix.col, matrix.data):
        f.write(f'{i + 1} {j + 1} {v}\n')
print('matrix.mtx has finished')
print('-------------------------------------------------')

matrix.mtx has finished
-------------------------------------------------


In [ ]:
df.set_index(f'bin{binsize}_pos', inplace=True)
xy_pos=df.loc[:,['x','y']]
xy_pos = xy_pos.drop_duplicates(subset=['x', 'y'])
xy_pos = xy_pos.sort_values(by=['x', 'y'], ascending=[True, True])
xy_pos.to_csv(f'{output_dir}/xy_pos.csv', index=True)